Load csv dataset and drop low minute players

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

MIN_MINUTES = 900 #10 full matches

plyrs24_25 = pd.read_csv("../data/raw/players_data-2024_2025.csv")
plyrs25_26 = pd.read_csv("../data/raw/players_data-2025_2026.csv")

plyrs24_25 = plyrs24_25[plyrs24_25['Min'] >= MIN_MINUTES]
plyrs25_26 = plyrs25_26[plyrs25_26['Min'] >= MIN_MINUTES]

# Unneccesary features that make things bloated
DROP_FEATURES = ['Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper',
                 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 
                'PK_stats_shooting', 'PKatt_stats_shooting', 'Rk_stats_playing_time', 'Nation_stats_playing_time', 'Pos_stats_playing_time', 
                'Comp_stats_playing_time', 'Age_stats_playing_time', 'Born_stats_playing_time', 'MP_stats_playing_time', 'Min_stats_playing_time',
                 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting',
                 '90s_stats_shooting', 'Gls_stats_shooting', 'Rk_stats_misc', 'Nation_stats_misc', 'Pos_stats_misc', 'Comp_stats_misc', 'Age_stats_misc', 
                'Born_stats_misc', '90s_stats_misc', 'CrdY_stats_misc', 'CrdR_stats_misc', '90s_stats_playing_time', 'Starts_stats_playing_time', 'Nation_stats_passing', 
                 'Pos_stats_passing', 'Comp_stats_passing', 'Nation_stats_passing_types', 'Pos_stats_passing_types', 'Comp_stats_passing_types', 'Nation_stats_gca', 
                 'Pos_stats_gca', 'Comp_stats_gca', 'Nation_stats_defense', 'Pos_stats_defense', 'Comp_stats_defense', 'Nation_stats_possession', 'Pos_stats_possession', 
                 'Comp_stats_possession', 'Nation_stats_keeper_adv', 'Pos_stats_keeper_adv', 'Comp_stats_keeper_adv'
                ]

plyrs24_25 = plyrs24_25.drop(columns=DROP_FEATURES)
plyrs25_26 = plyrs25_26.drop(columns=DROP_FEATURES)

plyrs24_25 = plyrs24_25.set_index('Player')
plyrs25_26 = plyrs25_26.set_index('Player')

metadata = ['Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born']

features = [col for col in plyrs24_25.columns if col not in metadata]

WEIGHT_24_25 = 0.3
WEIGHT_25_26 = 0.7

features_24_25 = plyrs24_25[features]
features_25_26 = plyrs25_26[features]

features_24_25, features_25_26 = features_24_25.align(features_25_26, join = "outer", axis=0)

features_24_25 = features_24_25.select_dtypes(include="number")
features_25_26 = features_25_26.select_dtypes(include="number")

combined_features = (WEIGHT_24_25 * features_24_25.fillna(0)) + (WEIGHT_25_26 * features_25_26.fillna(0))

players = combined_features.join(plyrs25_26[metadata], how = 'inner')

players = players.copy() # get rid of fragmented df warning, avoids bad performence
players = players.reset_index()

players['PlayerID'] = range(1, len(players) + 1)
players = players.set_index('PlayerID')

players.head()

,Player,#OPA,#OPA/90,+/-,+/-90,/90,1/3,1/3_stats_possession,2CrdY,90s,...,xG+/-,xG+/-90,xG+xAG,xG_stats_shooting,Nation,Pos,Squad,Comp,Age,Born
PlayerID,,,,,,,,,,,,,,,,,,,,,
1,Aaron Wan-Bissaka,0.0,0.000,-12.9,-0.878,0.00,46.4,29.3,0.0,18.34,...,-10.05,-0.645,0.084,0.36,cd COD,DF,West Ham,eng Premier League,28.0,1997.0
2,Aaron Zehnter,0.0,0.000,-0.7,-0.063,0.00,12.6,7.7,0.0,7.70,...,-3.08,-0.280,0.098,0.35,de GER,DF,Wolfsburg,de Bundesliga,21.0,2004.0
3,Aarón Escandell,25.9,1.365,-13.3,-0.700,0.28,4.2,0.0,0.0,13.30,...,-10.57,-0.560,0.000,0.00,es ESP,GK,Oviedo,es La Liga,30.0,1995.0
4,Aarón Martín,0.0,0.000,-7.5,-0.390,0.00,51.1,17.0,0.0,20.37,...,-2.92,-0.115,0.337,0.48,es ESP,DF,Genoa,it Serie A,28.0,1997.0
5,Abdoul Coulibaly,0.0,0.000,-5.6,-0.448,0.00,25.9,2.8,0.7,8.82,...,-3.92,-0.308,0.035,0.35,de GER,DF,Werder Bremen,de Bundesliga,18.0,2007.0


In [3]:
players.isna().sum()

Player     0
#OPA       0
#OPA/90    0
+/-        0
+/-90      0
          ..
Pos        0
Squad      0
Comp       0
Age        0
Born       0
Length: 210, dtype: int64

In [4]:
for col in players.columns:
    print(col + " ", end="")

Player #OPA #OPA/90 +/- +/-90 /90 1/3 1/3_stats_possession 2CrdY 90s 90s_stats_defense 90s_stats_gca 90s_stats_keeper_adv 90s_stats_passing 90s_stats_passing_types 90s_stats_possession A-xAG Age_stats_defense Age_stats_gca Age_stats_keeper_adv Age_stats_passing Age_stats_passing_types Age_stats_possession Ast Ast_stats_passing Att Att (GK) Att 3rd Att 3rd_stats_possession Att Pen Att_stats_defense Att_stats_keeper_adv Att_stats_passing_types Att_stats_possession AvgDist AvgLen Blocks Blocks_stats_defense Born_stats_defense Born_stats_gca Born_stats_keeper_adv Born_stats_passing Born_stats_passing_types Born_stats_possession CK CK_stats_keeper_adv CPA CS CS% Carries Clr Cmp Cmp% Cmp%_stats_keeper_adv Cmp_stats_keeper_adv Cmp_stats_passing_types Compl CrdR CrdY Crs CrsPA Crs_stats_misc D Dead Def Def 3rd Def 3rd_stats_possession Def Pen Dis Dist Err FK FK_stats_keeper_adv FK_stats_passing_types Fld Fld_stats_misc Fls G+A G+A-PK G-PK G-xG G/Sh G/SoT GA GA90 GA_stats_keeper_adv GCA GCA90 G

In [5]:
players['Comp'].unique().tolist()

['eng Premier League',
 'de Bundesliga',
 'es La Liga',
 'it Serie A',
 'fr Ligue 1']

In [9]:
OUTFIELD_FEATURES = ['90s', 'Gls', 'Ast', 'xG', 'xAG', 'npxG', 'G-PK', 
                     'Tkl', 'TklW','Blocks', 'Int', 'Clr', 'Err',
                     'PrgP', 'PrgC', 'KP', 'PPA', 
                     'Touches', 'Carries', 'PrgR', 'Mis', 'Dis']

In [11]:
def build_league_outfield_df(df, league_name):
    league_df = df[df['Comp'] == league_name]
    league_outfield = league_df[OUTFIELD_FEATURES]
    
    #adjust to a per 90 basis
    league_outfield = league_outfield.apply(lambda x : x / league_outfield['90s'])
    league_outfield = league_outfield.drop(columns='90s')

    #rename columns to reflect per 90 scale
    new_cols = {col:f'{col}/90' for col in league_outfield.columns.tolist()}
    league_outfield = league_outfield.rename(columns=new_cols)

    #Normalize using standard scaler
    scaler = StandardScaler()
    league_outfield = pd.DataFrame(scaler.fit_transform(league_outfield),
                                   columns=league_outfield.columns,
                                   index=league_outfield.index)
    
    return league_outfield

In [13]:
prem_outfield = build_league_outfield_df(players, 'eng Premier League')
laliga_outfield = build_league_outfield_df(players, 'es La Liga')
bundesliga_outfield = build_league_outfield_df(players, 'de Bundesliga')
serieA_outfield = build_league_outfield_df(players, 'it Serie A')
ligue1_outfield = build_league_outfield_df(players, 'fr Ligue 1')

outfield_vectors = pd.concat([prem_outfield, laliga_outfield, bundesliga_outfield, ligue1_outfield])

outfield_vectors.head()

,Gls/90,Ast/90,xG/90,xAG/90,npxG/90,G-PK/90,Tkl/90,TklW/90,Blocks/90,Int/90,...,Err/90,PrgP/90,PrgC/90,KP/90,PPA/90,Touches/90,Carries/90,PrgR/90,Mis/90,Dis/90
PlayerID,,,,,,,,,,,,,,,,,,,,,
1,-0.634691,-0.061989,-0.788242,-0.129285,-0.823774,-0.639188,0.091328,0.321312,1.587057,2.471454,...,0.096637,0.258571,0.835891,0.164207,0.508401,0.448726,-0.238954,0.469029,-0.261228,-0.220233
11,-0.846567,0.391051,-0.577097,1.669945,-0.581036,-0.877761,1.023429,0.846939,-0.340002,0.889462,...,-1.206920,1.600263,-0.298368,1.236280,1.188241,0.119578,-0.206280,-0.799154,-0.261082,0.033249
18,-0.681424,-0.370409,-0.647693,-0.176574,-0.672550,-0.691810,0.907655,0.853265,1.457931,0.586659,...,0.026834,0.261397,0.379812,0.249737,0.614685,0.685229,0.230862,1.062003,-0.339339,-0.064204
40,0.462109,0.820333,-0.240997,1.110261,-0.194646,0.595807,-0.361901,-0.390955,1.186852,-0.695374,...,-0.954432,1.503618,1.704643,1.394631,2.647441,0.189925,0.949565,0.933115,0.377532,0.311824
41,-0.434061,-1.012454,-0.341947,-0.094998,-0.310701,-0.413279,0.277748,0.667715,-0.257397,0.988342,...,-1.206920,-0.106569,0.287308,-0.306135,-0.498945,0.107740,-0.155562,-0.493325,0.088745,-0.065220


In [15]:
GK_FEATURES = ["90s", "GA90","Save%", "PSxG", "PSxG+/-", "PSxG/SoT", "SoTA", "Cmp%", "Launch%", "AvgLen", "AvgDist", "PrgDist",
                "PassLive", "PassDead", "1/3", "Att 3rd", "Def 3rd", "Def Pen", "Clr",]

In [20]:
def build_league_gk_df(df, league_name):
    league_df = df[df['Comp'] == league_name]
    league_gk = league_df[GK_FEATURES]

    cols_to_change = ['PSxG', 'PSxG+/-', 'SoTA', 'PrgDist', 'PassLive', 'PassDead', '1/3', 'Att 3rd','Def 3rd', 'Def Pen', 'Clr']
    league_gk[cols_to_change] = league_gk[cols_to_change].apply(lambda x : x / league_gk['90s'])

    new_cols = {col: f'{col}/90' for col in cols_to_change}
    league_gk = league_gk.rename(columns=new_cols)
    league_gk = league_gk.drop(columns='90s')

    scaler = StandardScaler()
    league_gk = pd.DataFrame(scaler.fit_transform(league_gk),
                            columns = league_gk.columns,
                            index = league_gk.index)
    return league_gk

In [22]:
prem_keepers = build_league_gk_df(players, 'eng Premier League')
laliga_keepers = build_league_gk_df(players, 'es La Liga')
bundesliga_keepers = build_league_gk_df(players, 'de Bundesliga')
serieA_keepers = build_league_gk_df(players, 'it Serie A')
ligue1_keepers = build_league_gk_df(players, 'fr Ligue 1')

gk_vectors = pd.concat([prem_keepers, laliga_keepers, bundesliga_keepers, serieA_keepers, ligue1_keepers])

gk_vectors.head()

,GA90,Save%,PSxG/90,PSxG+/-/90,PSxG/SoT,SoTA/90,Cmp%,Launch%,AvgLen,AvgDist,PrgDist/90,PassLive/90,PassDead/90,1/3/90,Att 3rd/90,Def 3rd/90,Def Pen/90,Clr/90
PlayerID,,,,,,,,,,,,,,,,,,
1,-0.300679,-0.307198,-0.304558,-0.080742,-0.307331,-0.305931,0.372749,-0.296845,-0.306975,-0.304403,-0.087560,0.435129,-0.056617,-0.191148,-0.644773,0.070782,-0.240283,0.957566
11,-0.300679,-0.307198,-0.304558,-0.080742,-0.307331,-0.305931,0.164427,-0.296845,-0.306975,-0.304403,0.340110,1.307369,2.610522,1.101701,0.063005,0.755808,-0.523635,-0.479402
18,-0.300679,-0.307198,-0.304558,-0.080742,-0.307331,-0.305931,0.359632,-0.296845,-0.306975,-0.304403,-0.157338,0.019837,-0.007277,-0.105375,1.353488,1.103140,-0.265749,0.479580
40,-0.300679,-0.307198,-0.304558,-0.080742,-0.307331,-0.305931,0.385094,-0.296845,-0.306975,-0.304403,-0.210680,1.480743,0.078413,0.490255,-0.568435,-0.307421,-0.621790,-0.905468
41,-0.300679,-0.307198,-0.304558,-0.080742,-0.307331,-0.305931,-1.450451,-0.296845,-0.306975,-0.304403,-0.751171,-0.291468,0.439926,-0.106698,0.881559,-0.467032,-0.280288,0.255904


In [24]:
outfield_vectors.to_csv("../data/clean/outfield_vectors.csv")
gk_vectors.to_csv("../data/clean/gk_vectors.csv")
players.to_csv("../data/clean/players.csv")

In [27]:
haaland = players[players['Player'] == 'Erling Haaland']
haaland_id = 236